# SKY Methods: Formal Algorithm Description

This notebook is an executable companion to the **Methods** section of the SKY paper.  
Each algorithm is reproduced in both pseudocode (Markdown) and working Python.  
All cells are offline-safe: no Materials Project API key is required.

**Table of contents**
1. Imports and configuration  
2. Algorithm 1 — MAGPIE composition featurisation  
3. Algorithm 2 — KNN retrieval with confidence calibration  
4. Algorithm 3 — Recursive synthesis search  
5. Dataset description


In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pymatgen.core import Composition

# Add project root to path if running from notebooks/
import os, pathlib
PROJECT_ROOT = pathlib.Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import ASSETS_DIR

print(f'Project root : {PROJECT_ROOT}')
print(f'Assets dir   : {ASSETS_DIR}')
print(f'numpy {np.__version__}')


## Algorithm 1 — MAGPIE Composition Featurisation

### Formal definition

Let $f : \mathcal{C} \to \mathbb{R}^{132}$ be the MAGPIE featuriser  
where $\mathcal{C}$ is the space of chemical compositions.  
For a composition $c \in \mathcal{C}$ the 132-dimensional feature vector is:

$$\mathbf{x} = f(c) = \bigl[\mathbf{x}^{\text{stoich}},\; \mathbf{x}^{\text{elem}},\; \mathbf{x}^{\text{ionic}},\; \mathbf{x}^{\text{elec}}\bigr]$$

### Feature groups

| Group | Dim | Description |
|-------|-----|-------------|
| **Stoichiometric** | 5 | Norm-0 through norm-10 of the stoichiometric coefficients. Captures elemental ratios without identifying which elements are present. |
| **Elemental statistics** | 65 | For each of 13 tabulated elemental properties (atomic number, atomic weight, period, group, electronegativity, covalent radius, valence electrons, unfilled *s/p/d/f* shells, NpUnfilled, GSvolume/atom, GSbandgap, GSmagmom) compute weighted mean, weighted MAD, range, min, max → 65 features total. |
| **Ionic compound features** | 8 | Whether the compound is ionic (and can charge-balance), maximum ionic character, mean ionic character, fraction with positive/negative oxidation states. Requires site oxidation state enumeration. |
| **Electronic structure** | 4 | Fraction of electrons in *s/p/d/f* orbitals for the mean valence electron configuration. |

**Reference**: Ward et al. (2016) *npj Computational Materials* 2, 16028.  
DOI: [10.1038/s41524-016-0005-2](https://doi.org/10.1038/s41524-016-0005-2)

```
Algorithm 1: MAGPIE featurisation
Input : Composition string s (e.g. 'LiCoO2')
Output: Feature vector x ∈ R^132

1.  Parse s into {element: fraction} via pymatgen.Composition
2.  For each group g ∈ {stoich, elem_stats, ionic, electronic}:
        x_g ← compute_group_features(composition, g)
3.  x ← concat(x_stoich, x_elem_stats, x_ionic, x_electronic)   # dim 132
4.  Return x
```


In [ ]:
# Load the pre-computed MAGPIE embedding matrix from the HDF5 asset file
import h5py

embedding_file = ASSETS_DIR / 'magpie_embeddings.h5'

if embedding_file.exists():
    with h5py.File(embedding_file, 'r') as hf:
        embeddings = hf['embeddings'][:]
        formulas   = [f.decode() if isinstance(f, bytes) else f
                      for f in hf['formulas'][:]]
    print(f'Embedding matrix shape : {embeddings.shape}')
    print(f'Feature dimension      : {embeddings.shape[1]}  (expected 132)')
    print(f'Number of materials    : {embeddings.shape[0]}')
    print(f'Example formulas       : {formulas[:5]}')
else:
    print(f'HDF5 file not found at {embedding_file}')
    print('Demonstrating feature shape with a single composition...')
    try:
        from matminer.featurizers.composition import ElementProperty
        featurizer = ElementProperty.from_preset('magpie')
        comp = Composition('LiCoO2')
        feats = featurizer.featurize(comp)
        print(f'  LiCoO2 MAGPIE vector shape: {np.array(feats).shape}')
    except ImportError:
        print('  matminer not installed — expected feature dimension is 132.')
    print('  (Run scripts/build_embeddings.py to generate the full HDF5 file.)')


## Algorithm 2 — KNN Retrieval with Confidence Calibration

### Formal definition

Given a query composition $c_q$ with feature vector $\mathbf{x}_q = f(c_q)$, the
$k$-nearest-neighbour retrieval returns a ranked list of materials together with
calibrated confidence scores.

**Distance metric**: Euclidean distance in MAGPIE space
$$d_i = \|\mathbf{x}_q - \mathbf{x}_i\|_2$$

**Confidence function** (bandwidth $\sigma > 0$):
$$\text{conf}(d_i, \sigma) = \exp\!\left(-\frac{d_i}{\sigma}\right) \in (0, 1]$$

The bandwidth $\sigma$ is chosen by minimising the Expected Calibration Error (ECE)
on a held-out validation split (see `src.evaluation.confidence_calibration`).

```
Algorithm 2: Confidence-calibrated KNN retrieval
Input : Query composition c_q, database D, integer k, bandwidth σ
Output: Ranked list [(material_i, d_i, conf_i)] for i = 1..k

1.  x_q ← MAGPIE(c_q)                        # Algorithm 1
2.  For each c_i ∈ D:
        d_i ← ||x_q - x_i||_2
3.  Sort D by d_i ascending → [c_(1), ..., c_(|D|)]
4.  For i = 1..k:
        conf_i ← exp(-d_(i) / σ)
5.  Return [(c_(i), d_(i), conf_(i))]_{i=1}^{k}
```


In [ ]:
from src.evaluation.confidence_calibration import SIGMA_GRID

print('Sigma sweep grid (SIGMA_GRID):', SIGMA_GRID)
print()

# Demonstrate the confidence function for different sigma values
distances = np.linspace(0, 20, 300)

fig, ax = plt.subplots(figsize=(8, 4))
for sigma in SIGMA_GRID:
    confidence = np.exp(-distances / sigma)
    ax.plot(distances, confidence, label=f'σ = {sigma}')

ax.set_xlabel('Euclidean distance d', fontsize=12)
ax.set_ylabel('conf = exp(−d / σ)', fontsize=12)
ax.set_title('Confidence function for SIGMA_GRID values', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4, label='conf = 0.5')
ax.set_xlim(0, 20)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('confidence_sigma_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confidence at d=5 for each sigma:')
for sigma in SIGMA_GRID:
    print(f'  sigma={sigma:5.1f}  conf={np.exp(-5.0/sigma):.4f}')


## Algorithm 3 — Recursive Synthesis Search

When no direct synthesis recipe exists for a target material, SKY performs a
confidence-guided depth-first search over the KNN graph.

### Formal pseudocode

```
Algorithm 3: RecursiveSynthesisSearch(target, max_depth, min_conf, σ, k)
Input : target composition c_0
        max_depth  D  (default 3)
        min_confidence  θ  (default 0.70)
        bandwidth  σ
        neighbours per level  k
Output: ranked list of (recipe, confidence, source_material, path_length)

candidates ← []
visited    ← {}

procedure SEARCH(node, depth, conf_threshold):
    if depth ≥ D  OR  node.confidence < θ  OR  node ∈ visited:
        return
    visited ← visited ∪ {node}
    recipes ← GetRecipes(node.formula)           # MP synthesis database look-up
    for r ∈ recipes:
        candidates.append((r, node.confidence, node.formula, depth))
    neighbours ← KNN(node.formula, k)            # Algorithm 2
    for n ∈ neighbours where n.confidence ≥ conf_threshold × confidence_decay:
        SEARCH(n, depth + 1, conf_threshold × confidence_decay)

SEARCH(root(c_0), depth=0, conf_threshold=1.0)
return Sort(candidates, key=confidence / (1 + path_penalty × path_length))
```

### Complexity

In the worst case (no early termination), the search tree has at most
$O(k^{D})$ nodes, each requiring one MP API call.  
With default parameters $(k=10, D=3)$ this is **at most 1 000 API calls**,
but the confidence threshold $\theta$ and the `visited` set ensure typical
runs explore far fewer nodes.

### Termination conditions

| Condition | Effect |
|-----------|--------|
| `depth >= max_depth` | Hard cutoff — prevents unbounded recursion |
| `node.confidence < min_confidence` | Prunes low-quality branches early |
| `node in visited` | Cycle detection — avoids revisiting materials |


## Dataset Description

SKY is trained and evaluated on synthesis recipes extracted from the
[Materials Project](https://materialsproject.org) synthesis descriptions database.
The dataset is distributed as a gzip-compressed JSON file:
`assets/mp_synthesis_recipes.json.gz`.

| Field | Description |
|-------|-------------|
| `material_id` | MP identifier (e.g. `mp-19770`) |
| `reduced_formula` | Reduced stoichiometric formula |
| `synthesis_type` | Method family: `solid-state`, `hydrothermal`, `sol-gel`, etc. |
| `precursor_elements` | List of element symbols present in precursors |
| `paragraph_string` | Raw extracted synthesis paragraph |
| `temperature_celsius` | Reported sintering/reaction temperature (may be `null`) |


In [ ]:
import gzip
import json
from collections import Counter

dataset_path = ASSETS_DIR / 'mp_synthesis_recipes.json.gz'

try:
    with gzip.open(dataset_path, 'rt', encoding='utf-8') as fh:
        recipes = json.load(fh)
    print(f'Loaded {len(recipes):,} synthesis recipes.')

    # Synthesis type distribution
    type_counts = Counter(r.get('synthesis_type', 'unknown') for r in recipes)
    print('\nSynthesis type distribution:')
    for stype, count in type_counts.most_common():
        bar = '#' * (count * 40 // max(type_counts.values()))
        print(f'  {stype:<20} {count:>6,}  {bar}')

    # Temperature distribution
    temps = [r['temperature_celsius'] for r in recipes
             if r.get('temperature_celsius') is not None]
    if temps:
        temps = np.array(temps)
        print(f'\nTemperature statistics (n={len(temps):,}):')
        print(f'  min  = {temps.min():.0f} °C')
        print(f'  mean = {temps.mean():.0f} °C')
        print(f'  max  = {temps.max():.0f} °C')

except FileNotFoundError:
    print(f'Dataset file not found: {dataset_path}')
    print('This is expected in offline/CI environments.')
    print('Run scripts/download_mp_recipes.py to fetch the dataset.')
    print()
    print('Expected schema example:')
    example = {
        'material_id': 'mp-19770',
        'reduced_formula': 'LiCoO2',
        'synthesis_type': 'solid-state',
        'precursor_elements': ['Li', 'Co'],
        'paragraph_string': 'LiOH and CoCO3 were mixed and fired at 850 °C for 24 h in air.',
        'temperature_celsius': 850,
    }
    print(json.dumps(example, indent=2))
except json.JSONDecodeError as e:
    print(f'JSON decode error: {e}')
